# Attention U-Net

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from tqdm.auto import tqdm
except ImportError:
    class _SimpleTQDM:
        def __init__(self, iterable, desc=None):
            self.iterable = iterable
            self.desc = desc

        def __iter__(self):
            return iter(self.iterable)

        def set_postfix(self, *args, **kwargs):
            pass

    def tqdm(iterable, desc=None):
        return _SimpleTQDM(iterable, desc=desc)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_size = (512, 384)
batch_size = 1
num_epochs = 10
learning_rate = 1e-4

torch.manual_seed(42)
random.seed(42)

print("device:", device)


## Data

In [ ]:
isic_slug = "tschandl/isic2018-challenge-task1-data-segmentation"
local_isic_root = Path("U-Nets/data/isic2018")
kaggle_input_root = Path("/kaggle/input/isic2018-challenge-task1-data-segmentation")

if kaggle_input_root.exists():
    isic_root = kaggle_input_root
elif local_isic_root.exists() and list(local_isic_root.rglob("*.jpg")):
    isic_root = local_isic_root
else:
    import kagglehub

    downloaded_path = Path(kagglehub.dataset_download(isic_slug))
    isic_root = downloaded_path

print("ISIC root:", isic_root)


In [ ]:
image_paths = sorted(isic_root.rglob("*.jpg"))
mask_paths = sorted(isic_root.rglob("*_segmentation.png"))

mask_by_id = {
    path.name.replace("_segmentation.png", ""): path
    for path in mask_paths
}

samples = []

for image_path in image_paths:
    mask_path = mask_by_id.get(image_path.stem)

    if mask_path is not None:
        samples.append((image_path, mask_path))

print("images found:", len(image_paths))
print("masks found:", len(mask_paths))
print("paired samples:", len(samples))

if len(samples) == 0:
    raise RuntimeError("No ISIC image/mask pairs found. Check the Kaggle dataset path or download step.")


In [ ]:
class ISICSegmentationDataset(Dataset):
    def __init__(self, samples, image_size=(512, 384)):
        self.samples = samples
        self.image_size = image_size

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, mask_path = self.samples[idx]

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image = TF.resize(image, self.image_size, interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.image_size, interpolation=InterpolationMode.NEAREST)

        image = TF.to_tensor(image)
        mask = torch.as_tensor(np.array(mask), dtype=torch.float32)
        mask = (mask > 127).float().unsqueeze(0)

        return image, mask


full_dataset = ISICSegmentationDataset(samples, image_size=image_size)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_seg_dataset, test_seg_dataset = random_split(
    full_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_seg_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_seg_dataset, batch_size=batch_size, shuffle=False)

images, masks = next(iter(train_loader))
print("images:", images.shape)
print("masks:", masks.shape)
print("mask values:", masks.unique())


## Model

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.down1 = DoubleConv(3, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)

        self.bottleneck = DoubleConv(512, 1024)

    def forward(self, x):
        skip1 = self.down1(x)
        x = self.pool(skip1)

        skip2 = self.down2(x)
        x = self.pool(skip2)

        skip3 = self.down3(x)
        x = self.pool(skip3)

        skip4 = self.down4(x)
        x = self.pool(skip4)

        latent = self.bottleneck(x)

        return latent, skip1, skip2, skip3, skip4


In [ ]:
class AttentionGate(nn.Module):
    def __init__(self, skip_channels, gating_channels, hidden_channels):
        super().__init__()

        self.skip_projection = nn.Conv2d(skip_channels, hidden_channels, kernel_size=1)
        self.gating_projection = nn.Conv2d(gating_channels, hidden_channels, kernel_size=1)
        self.attention = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, skip, gating):
        if gating.shape[-2:] != skip.shape[-2:]:
            gating = F.interpolate(gating, size=skip.shape[-2:], mode="bilinear", align_corners=False)

        skip_features = self.skip_projection(skip)
        gating_features = self.gating_projection(gating)
        alpha = self.attention(skip_features + gating_features)

        return skip * alpha, alpha


In [ ]:
class AttentionUpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.attention_gate = AttentionGate(skip_channels, out_channels, out_channels // 2)
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        filtered_skip, attention_map = self.attention_gate(skip, x)
        x = torch.cat([filtered_skip, x], dim=1)
        x = self.conv(x)

        return x, attention_map


In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.up1 = AttentionUpBlock(1024, 512, 512)
        self.up2 = AttentionUpBlock(512, 256, 256)
        self.up3 = AttentionUpBlock(256, 128, 128)
        self.up4 = AttentionUpBlock(128, 64, 64)

        self.final_conv = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, latent, skip1, skip2, skip3, skip4):
        x, attention4 = self.up1(latent, skip4)
        x, attention3 = self.up2(x, skip3)
        x, attention2 = self.up3(x, skip2)
        x, attention1 = self.up4(x, skip1)

        logits = self.final_conv(x)
        attention_maps = [attention1, attention2, attention3, attention4]

        return logits, attention_maps


In [ ]:
class AttentionUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = Encoder()
        self.decoder = AttentionDecoder()

    def forward(self, x, return_attention=False):
        latent, skip1, skip2, skip3, skip4 = self.encoder(x)
        logits, attention_maps = self.decoder(latent, skip1, skip2, skip3, skip4)

        if return_attention:
            return logits, attention_maps

        return logits


In [ ]:
model = AttentionUNet().to(device)

x = torch.randn(1, 3, 512, 384, device=device)

with torch.no_grad():
    logits, attention_maps = model(x, return_attention=True)

print("logits:", logits.shape)

for idx, attention_map in enumerate(attention_maps, start=1):
    print(f"attention{idx}:", attention_map.shape)


## Loss and training

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


def dice_loss_from_logits(logits, masks, eps=1e-7):
    probs = torch.sigmoid(logits)

    intersection = (probs * masks).sum(dim=(1, 2, 3))
    probs_sum = probs.sum(dim=(1, 2, 3))
    masks_sum = masks.sum(dim=(1, 2, 3))

    dice = (2 * intersection + eps) / (probs_sum + masks_sum + eps)

    return 1 - dice.mean()


In [ ]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    total_bce_loss = 0
    total_dice_loss = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}")

    for images, masks in progress_bar:
        images = images.to(device)
        masks = masks.to(device).float()

        logits = model(images)

        bce_loss = loss_fn(logits, masks)
        dice_loss = dice_loss_from_logits(logits, masks)
        loss = bce_loss + dice_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_bce_loss += bce_loss.item()
        total_dice_loss += dice_loss.item()

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "bce": f"{bce_loss.item():.4f}",
            "dice": f"{dice_loss.item():.4f}",
        })

    avg_loss = total_loss / len(train_loader)
    avg_bce_loss = total_bce_loss / len(train_loader)
    avg_dice_loss = total_dice_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs}, "
        f"Loss: {avg_loss:.4f}, "
        f"BCE: {avg_bce_loss:.4f}, "
        f"Dice: {avg_dice_loss:.4f}"
    )


## Test

In [ ]:
def dice_score(pred_mask, true_mask, eps=1e-7):
    pred_mask = pred_mask.float()
    true_mask = true_mask.float()

    intersection = (pred_mask * true_mask).sum(dim=(1, 2, 3))
    pred_sum = pred_mask.sum(dim=(1, 2, 3))
    true_sum = true_mask.sum(dim=(1, 2, 3))

    dice = (2 * intersection + eps) / (pred_sum + true_sum + eps)
    return dice.mean()


def iou_score(pred_mask, true_mask, eps=1e-7):
    pred_mask = pred_mask.float()
    true_mask = true_mask.float()

    intersection = (pred_mask * true_mask).sum(dim=(1, 2, 3))
    union = pred_mask.sum(dim=(1, 2, 3)) + true_mask.sum(dim=(1, 2, 3)) - intersection

    iou = (intersection + eps) / (union + eps)
    return iou.mean()


In [ ]:
model.eval()

images, masks = next(iter(test_loader))
images = images.to(device)
masks = masks.to(device).float()

with torch.no_grad():
    logits, attention_maps = model(images, return_attention=True)
    probs = torch.sigmoid(logits)
    pred_masks = (probs > 0.5).float()

print("images:", images.shape)
print("logits:", logits.shape)
print("probability range:", probs.min().item(), probs.max().item())
print("dice:", dice_score(pred_masks, masks).item())
print("iou:", iou_score(pred_masks, masks).item())


In [ ]:
if plt is None:
    print("matplotlib is not installed, so plotting is skipped.")
else:
    num_samples = 4

    sample_images = []
    sample_masks = []
    sample_probs = []
    sample_pred_masks = []
    sample_attention_maps = []

    model.eval()

    with torch.no_grad():
        for batch_images, batch_masks in test_loader:
            batch_images = batch_images.to(device)
            batch_masks = batch_masks.to(device).float()

            batch_logits, batch_attention_maps = model(batch_images, return_attention=True)
            batch_probs = torch.sigmoid(batch_logits)
            batch_pred_masks = (batch_probs > 0.5).float()

            sample_images.append(batch_images[0].detach().cpu())
            sample_masks.append(batch_masks[0].detach().cpu())
            sample_probs.append(batch_probs[0].detach().cpu())
            sample_pred_masks.append(batch_pred_masks[0].detach().cpu())
            sample_attention_maps.append(batch_attention_maps[0][0].detach().cpu())

            if len(sample_images) == num_samples:
                break

    rows = len(sample_images)
    cols = 5
    titles = ["Image", "True mask", "Probability", "Pred mask", "Attention"]

    plt.figure(figsize=(16, 3.5 * rows))

    for row in range(rows):
        panels = [
            sample_images[row].permute(1, 2, 0),
            sample_masks[row][0],
            sample_probs[row][0],
            sample_pred_masks[row][0],
            sample_attention_maps[row][0],
        ]

        for col, panel in enumerate(panels):
            plt.subplot(rows, cols, row * cols + col + 1)

            if col == 0:
                plt.imshow(panel)
            else:
                plt.imshow(panel, cmap="gray")

            if row == 0:
                plt.title(titles[col])

            plt.axis("off")

    plt.tight_layout()
    plt.show()
